In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.CutMasks.MaskUtils import *

from makedf.mcstat import get_MCstat_unc

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/lpelegri/syst/"

today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics-other-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load df

In [ ]:
use_Ar23p = True

pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

if not use_Ar23p:
    mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV.df", keys2load, 100)
else:
    mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p.df", keys2load, 100)
    
mc_evt_df = mc_bnb_df['cc1pi']
mc_nu_df = mc_bnb_df['nudf']
mc_hdr_df = mc_bnb_df['hdr']

#Add weight column
data_tot_pot = 5.947e+18
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_evt_df))

#Do truth matchign
if not use_Ar23p:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_nu_df)
else:
    mc_evt_df = perform_truth_matching_low_memmory(mc_evt_df, mc_nu_df)


if not use_Ar23p:
    mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))
else:
    new_columns = []
    for c in mc_nu_df.columns:
        new_columns.append(('truth',) + c + ('',) + ('',))  # prepend 'truth'
    mc_nu_df.columns = pd.MultiIndex.from_tuples(new_columns)
    mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))

# Perform selection

In [ ]:
plot_sideband = False
mc_cumulative_masks = build_event_cumulative_masks(mc_evt_df, plot_sideband = plot_sideband)
mc_evt_df = mc_evt_df[mc_cumulative_masks["energy"]]

In [ ]:
#make it a slc df
mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_evt_df = mc_evt_df.sort_index()


In [ ]:
print("==== breakdown of selected events ====")
HelperFunctions.print_purity(mc_evt_df, ('truth','nu_categ','','','',''))

#print(mc_evt_df.genie_categ.value_counts())

# MC Stat

In [ ]:
import hashlib
def get_MCstat_unc(evt_df, hdr_df, n_universes=100):
    # Create a unique seed based on event metadata
    # Using a hash function that's deterministic
    meta_seeds = []
    for i in tqdm(range(len(evt_df))):
        this_hdr_df = hdr_df.loc[evt_df.reset_index(level=[2]).index[i]]
        runno = this_hdr_df.run
        subrunno = this_hdr_df.subrun
        evtno = this_hdr_df.evt
        slcid = mc_evt_df.loc[mc_evt_df.index[i]].slc.self
        seed_string = f"run_{runno}_subrun_{subrunno}_evt_{evtno}_slcid_{slcid}"
        #unique_seed = hash(f"run_{runno}_subrun_{subrunno}_evt_{evtno}_slcid_{slcid}") % (2**32)  # Ensure it's a 32-bit integer
        unique_seed = int(
            hashlib.sha256(seed_string.encode()).hexdigest(),
            16
        ) % (2**32)
        if unique_seed in meta_seeds:
            print("duplicate seed found", unique_seed)
            break
        meta_seeds.append(unique_seed)

    # make sure the seeds are unique!
    assert len(meta_seeds) == len(set(meta_seeds))

    # generate universes
    MCstat_univ_events = np.zeros((n_universes, len(evt_df)))
    poisson_mean = 1.0

    # get Poisson weights and save to "MCstat.univ_"
    # dummy df to hold the weights -- iterative inserting causes PerformanceWarning
    mcstat_univ_cols = pd.MultiIndex.from_product(
        [["truth"], ["MCstat"], [f"univ_{i}" for i in range(n_universes)],[""],[""],[""]],
    )
    mcstat_univ_wgt = pd.DataFrame(
        1.0,
        index=evt_df.index,
        columns=mcstat_univ_cols,
    )

    for uidx in range(n_universes):
        universe_string = f"universe_{uidx}"
        universe_seed = int(
            hashlib.sha256(universe_string.encode()).hexdigest(),
            16
        ) % (2**32)
            
        poisson_weights = []
        for sidx, meta_seed in enumerate(meta_seeds):
            # Combine universe seed with event seed for unique randomness -- per event, per universe
            combined_seed = (universe_seed + meta_seed) % (2**32)
            np.random.seed(combined_seed)
            
            poisson_val = np.random.poisson(poisson_mean)
            poisson_weights.append(poisson_val)
            
        mcstat_univ_wgt[("truth","MCstat", "univ_{}".format(uidx),'','','')] = np.array(poisson_weights)
        MCstat_univ_events[uidx, :] = np.array(poisson_weights)

    evt_df = evt_df.join(mcstat_univ_wgt)
    return evt_df, MCstat_univ_events

In [ ]:
mc_evt_df, MCstat_univ_events = get_MCstat_unc(mc_evt_df, mc_hdr_df, n_universes=100)

In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"

os.makedirs(file_dir, exist_ok=True)  # create directory if needed
show_plots = False

var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]
syst_name = "MCstat"

syst_dict = {}
for var_config in var_configs:

    univ_events, cv_events = get_univ_rates(cov_type="rate", 
                                            evtdf=mc_evt_df,
                                            var_config=var_config,
                                            n_univ=100,
                                            bkgd_subtract=True,
                                            syst_name=syst_name)
    
    ret_MCstat = get_covariance_matrix(univ_events, cv_events)
    syst_dict[var_config.var_save_name] = ret_MCstat["cov_frac"]

    if show_plots: 
        plot_univ_hists(univ_events, cv_events, syst_name, var_config, use_bin_width = True)
    
        frac_unc = (np.sqrt(np.diag(ret_MCstat["cov_frac"])), "MC Stats")
        plot_frac_unc([frac_unc], var_config)
    
        matrix_type = "cov"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_MCstat[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                     save_fig=save_fig, save_name=save_fig_name)
        
        matrix_type = "cov_frac"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_MCstat[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "FractionalCovariance"],
                     save_fig=save_fig, save_name=save_fig_name)
     
   
# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    if not use_Ar23p:
        np.savez(file_dir + "/mcstat_syst_dict.npz", **syst_dict)
    else:
        np.savez(file_dir + "/mcstat_syst_dict_ar23p.npz", **syst_dict)
    


# Flux

In [ ]:
syst_name = "Flux"

show_plots = False

syst_dict_flux_xsec = {}
syst_dict_flux_rate = {}
for var_config in var_configs:
    cov_type = "rate"
    
    univ_events, cv_events = get_univ_rates(cov_type=cov_type, 
                                            evtdf=mc_evt_df,
                                            var_config=var_config,
                                            n_univ=100,
                                            bkgd_subtract=True,
                                            syst_name=syst_name)
    
    ret_flux_rate = get_covariance_matrix(univ_events, cv_events)
    syst_dict_flux_rate[var_config.var_save_name] = ret_flux_rate["cov_frac"]

    if show_plots: 
        plot_univ_hists(univ_events, cv_events, syst_name, var_config, use_bin_width = True)
        
        frac_unc = (np.sqrt(np.diag(ret_flux_rate["cov_frac"])), "Flux")
        plot_frac_unc([frac_unc], var_config)
    
        matrix_type = "cov"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_flux_rate[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                     save_fig=save_fig, save_name=save_fig_name)
        
        matrix_type = "cov_frac"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_flux_rate[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
                     save_fig=save_fig, save_name=save_fig_name)
        
        matrix_type = "corr"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_flux_rate[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
                     save_fig=save_fig, save_name=save_fig_name)

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    if not use_Ar23p:
        np.savez(file_dir + "/flux_syst_dict_rate.npz", **syst_dict_flux_rate)
    else:
        np.savez(file_dir + "/flux_syst_dict_rate_ar23p.npz", **syst_dict_flux_rate)

    

# G4

In [ ]:
syst_name = "G4"

show_plots = False

syst_dict_g4_xsec = {}
syst_dict_g4_rate = {}
for var_config in var_configs:
    cov_type = "rate"
    univ_events, cv_events = get_univ_rates(cov_type=cov_type, 
                                            evtdf=mc_evt_df,
                                            var_config=var_config,
                                            n_univ=100,
                                            bkgd_subtract=True,
                                            syst_name=syst_name)
    
    ret_G4_rate = get_covariance_matrix(univ_events, cv_events)
    syst_dict_g4_rate[var_config.var_save_name] = ret_G4_rate["cov_frac"]
    
    if show_plots: 
        plot_univ_hists(univ_events, cv_events, syst_name, var_config, use_bin_width = True)

        frac_unc = (np.sqrt(np.diag(ret_G4_rate["cov_frac"])), "G4")
        plot_frac_unc([frac_unc], var_config)

        matrix_type = "cov_frac"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_G4_rate[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                     save_fig=save_fig, save_name=save_fig_name)

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    if not use_Ar23p:
        np.savez(file_dir + "/g4_syst_dict_rate.npz", **syst_dict_g4_rate)
    else:
        np.savez(file_dir + "/g4_syst_dict_rate_ar23p.npz", **syst_dict_g4_rate)

# GENIE

In [ ]:
syst_name = "GENIE"

show_plots = True

syst_dict_rate = {}
syst_dict_xsec = {}
for var_config in var_configs:
    cov_type = "xsec"
    univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
    ret_genie_xsec = get_covariance_matrix(univ_events, cv_events)
    print(cv_events)
    print(univ_events)
    cov_type = "rate"
    univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
    ret_genie_rate = get_covariance_matrix(univ_events, cv_events)
    if show_plots:
        signal_hists(evtdf=mc_evt_df, 
             nudf=mc_nu_df, 
             var_config=var_config, 
             save_fig=False, 
             save_name=None)

        plot_univ_hists(univ_events, cv_events, syst_name, var_config, use_bin_width = True)
        plot_univ_hists(univ_events, cv_events, syst_name, var_config, use_bin_width = True)

        
        matrix_type = "cov"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1], matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_genie_xsec[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                     save_fig=save_fig, save_name=save_fig_name)
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1]+"_rate", matrix_type)
        plot_heatmap(ret_genie_rate[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
                     save_fig=save_fig, save_name=save_fig_name)
        
        matrix_type = "cov_frac"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1], matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_genie_xsec[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
                     save_fig=save_fig, save_name=save_fig_name)
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1]+"_rate", matrix_type)
        plot_heatmap(ret_genie_rate[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
                     save_fig=save_fig, save_name=save_fig_name)
        
        matrix_type = "corr"
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1], matrix_type)
        title = "{} {}".format(syst_name, matrix_type)
        plot_heatmap(ret_genie_xsec[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
                     save_fig=save_fig, save_name=save_fig_name)
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name[1]+"_rate", matrix_type)
        plot_heatmap(ret_genie_rate[matrix_type], 
                     var_config.bins, 
                     plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
                     save_fig=save_fig, save_name=save_fig_name)

        # Create the named list
        frac_unc_named_list = [
            (np.sqrt(np.diag(ret_genie_xsec["cov_frac"])), "GENIE x-sec"),
            (np.sqrt(np.diag(ret_genie_rate["cov_frac"])), "GENIE rate")
        ]
        
        # Call the function
        plot_frac_unc(frac_unc_named_list, var_config)

    syst_dict_rate[var_config.var_save_name] = ret_genie_rate["cov_frac"]
    syst_dict_xsec[var_config.var_save_name] = ret_genie_xsec["cov_frac"]

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    if not use_Ar23p:
        np.savez(file_dir + "/genie_syst_dict_xsec.npz", **syst_dict_xsec)
    else:
        np.savez(file_dir + "/genie_syst_dict_xsec_ar23p.npz", **syst_dict_xsec)

    if not  use_Ar23p:
        np.savez(file_dir + "/genie_syst_dict_rate.npz", **syst_dict_rate)
    else:
        np.savez(file_dir + "/genie_syst_dict_rate_Ar23p.npz", **syst_dict_rate)

# All Uncertanties

In [ ]:
Total_Covariance_Frac = ret_Flux["cov_frac"] + ret_genie_rate["cov_frac"] + ret_MCstat["cov_frac"] + ret_G4["cov_frac"]

frac_unc_named_list = [
    (np.sqrt(np.diag(Total_Covariance_Frac)), "Total"),
    (np.sqrt(np.diag(ret_Flux["cov_frac"])), "Flux"),
    (np.sqrt(np.diag(ret_genie_xsec["cov_frac"])), "GENIE"),
    (np.sqrt(np.diag(ret_MCstat["cov_frac"])), "MC Stat"),
    (np.sqrt(np.diag(ret_G4["cov_frac"])), "G4"),
]

plot_frac_unc(frac_unc_named_list, var_config)